# Опыт 08: собственный BPE 4096, прежний Transformer

**Гипотеза:** объединение частых фрагментов текста сократит последовательности и ошибки формы кода, особенно отступов. Меняем токенизацию; сохраняем 20 000 исходных пар, 2+2 слоя / ширину128 / FFN512-ReLU / Post-LN, learned positions, FP32, AdamW LR0.001, batch8, seed42. Обучаем модель **с нуля**, старые веса загружать не нужно. Словарь увеличит embedding/head: **1 254 915 → 2 241 024 параметра**; этот эффект не отделён от BPE.

1. Открой **новую сессию Colab с T4**, загрузи этот notebook и выполни ячейки по порядку. Первая ячейка ставит только `tokenizers==0.23.2` и PyArrow.
2. Проверяются неизменные train/validation hashes, обратимость токенизации и KV-кеш. BPE обучается только на прежних 20 000 train-парах; исходные тексты не переписываются.
3. Обучение до **50 000 шагов**; контроли **2k / 5k / 10k / 20k / 30k / 50k**. Каждые 1000 сохраняется `latest.pt`; контроли получают отдельные checkpoints. Точные validation64/train64 прежние; ещё 192 heldout-вопроса оцениваются в конце.
4. В конце отдельно сравниваются validation64 с cap512 **BPE-токенов** и пределом **512 выходных байтов**. На всех 1919 test-вопросах измеряется только кешированный вариант cap512 BPE-токенов. Полный таймер включает encode/переносы/generation/decode/AST, исключает скачивание и loss.
5. Последняя ячейка скачает **`hw02-bpe4k-20k-results.zip`**: результаты, tokenizer, checkpoints, все test-генерации, manifest. Дополнительно **Файл → Скачать → Скачать IPYNB**, чтобы сохранить outputs. Верни ZIP и IPYNB.

Если остановил обучение или сработала проверка, **не запускай train-ячейку заново**: выполни последнюю ячейку скачивания и сохрани IPYNB. Checkpoint содержит optimizer/RNG/tokenizer для последующего продолжения. Colab может оборвать процесс до выполнения `finally`; тогда остаётся последний периодический checkpoint.

Оцениваются BPE-loss, AST, `solve`/return, отступы, EOS/cap, длины и точное совпадение с одним reference. **Код программ не исполняется, pass@1 не измеряется.** Loss byte/BPE напрямую не сравниваем. Качество на 10k сравниваем с byte10k, итоговое — с byte50k; одинаковое число шагов при batch8 означает одинаковые предъявления пар, но другое число токенов и стоимость.

Источники: [неизменное ДЗ](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [данные](https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/blob/b2724689ff079e9c22efd548fe134bfb29036dbb/README.md), [CodeT5](https://arxiv.org/html/2109.00859v1), [Tokenizers ByteLevel](https://huggingface.co/docs/tokenizers/v0.23.2/api/pre-tokenizers#tokenizers.pre_tokenizers.ByteLevel), [KV-кеш](https://huggingface.co/docs/transformers/main/cache_explanation). Число4k и бюджет — наш опыт, а не копия размера CodeT5.


In [ ]:
%pip install -q tokenizers==0.23.2 pyarrow


## 1. Настройки и сохранение результатов


In [ ]:
import ast
import hashlib
import json
import math
import os
import random
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 20000, 8
EVALUATION_STEPS = (2000, 5000, 10000, 20000, 30000, 50000)
TOTAL_STEPS, LOG_EVERY, CHECKPOINT_EVERY = 50000, 50, 1000
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, TOKEN_OFFSET, TARGET_VOCAB = 0, 1, 2, 3, 4096
GENERATION_CAP, INFERENCE_BATCH, BYTE_BUDGET = 512, 32, 512
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-4
RUN_DIR = Path('hw02-bpe4k-20k')
RESULTS_FILE = RUN_DIR / 'results.json'
TOKENIZER_FILE = RUN_DIR / 'tokenizer.json'
LATEST_CHECKPOINT = RUN_DIR / 'latest.pt'
ARCHIVE_FILE = Path('hw02-bpe4k-20k-results.zip')
assert tokenizers.__version__ == '0.23.2', 'Run the installation cell in a fresh session'
assert torch.cuda.is_available(), 'Включи GPU в настройках Colab'
assert not RESULTS_FILE.exists() and not LATEST_CHECKPOINT.exists(), 'Сначала сохрани прежний запуск; используй новую сессию'
RUN_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')
random.seed(SEED)
torch.manual_seed(SEED)

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()
def sha(raw): return hashlib.sha256(raw).hexdigest()
def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())
def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)
def save_results(): write_json(RESULTS_FILE, result)

result = {'experiment': '08-bpe4k-20k-pairs', 'phase': 'setup', 'actual_steps': 0,
    'runtime': {'device': str(device), 'gpu': torch.cuda.get_device_name(0),
                'torch': torch.__version__, 'tokenizers': tokenizers.__version__},
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'run_finished': False, 'test_evaluated': False, 'functional_quality_evaluated': False,
    'historical_byte_50k': {'provenance': 'saved user T4 result06/07; not measured in experiment08',
        'parameters': 1254915, 'validation64': {'examples': 64, 'ast': 41, 'return': 40, 'eos': 61, 'exact': 0},
        'test1919': {'examples': 1919, 'ast': 1088, 'return': 1060, 'eos': 1820,
                     'token_cap_without_eos': 99, 'indentation_errors': 754,
                     'cached_seconds_rounded': 61.40}}}
save_results()
print('RUNTIME', json.dumps(result['runtime']))


## 2. Прежние 20 000 train-пар и контрольные вопросы


In [ ]:
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
train_file = cache / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists(): urlretrieve(f'{base_url}/data/train/part-00000.parquet', train_file)
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, train_row_indices, validation = [], [], None
heldout_pool, heldout_questions = [], set()
offset = 0
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
        heldout_questions.update(r['question'] for r in validation)
    for index, row in enumerate(rows):
        if not eligible(row): continue
        if is_validation(row['question']):
            if row['question'] not in heldout_questions and len(heldout_pool) < 192:
                heldout_pool.append(row)
                heldout_questions.add(row['question'])
            continue
        train.append(row)
        train_row_indices.append(offset + index)
        if len(train) == TRAIN_SIZE: break
    if len(train) == TRAIN_SIZE: break
    offset += len(rows)
assert len(train) == 20000 and len(validation) == 64 and len(heldout_pool) == 192
train_hash, validation_hash = data_hash(train), data_hash(validation)
assert train_hash == '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
assert data_hash(train[:512]) == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert validation_hash == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert not ({r['question'] for r in train} & heldout_questions)
assert all(not is_validation(r['question']) for r in train)
train_diagnostic = train[:64]
validation_extra = heldout_pool
result['data'] = {'revision': revision, 'shard': 0, 'training_examples': len(train),
    'unique_train_questions': len({r['question'] for r in train}),
    'train_sha256': train_hash, 'validation64_sha256': validation_hash,
    'train_row_indices_sha256': data_hash(train_row_indices),
    'extra_heldout_examples': len(validation_extra), 'extra_heldout_sha256': data_hash(validation_extra),
    'extra_selection': 'first 192 distinct eligible hash-heldout questions after excluding original validation64',
    'question_overlap_train_heldout': 0, 'download_and_selection_seconds': time.perf_counter() - data_started,
    'selection': 'unchanged first20000 non-heldout rows of shard0; question<=512/code<512 UTF8 bytes'}
result['phase'] = 'data_ready'
save_results()
print('DATA', json.dumps(result['data']))


## 3. Собственный обратимый BPE и длины

BPE имеет **4093 обычных токена**; в модели добавлены PAD/BOS/EOS с IDs0/1/2. BPE IDs сдвигаются на3. Поэтому строки вроде `<pad>` внутри исходного кода остаются обычным текстом. Normalizer отсутствует; ByteLevel не добавляет ведущий пробел. Проверяются все train/heldout-тексты и отдельные сложные строки.


In [ ]:
tokenizer_started = time.perf_counter()
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, trim_offsets=False, use_regex=True)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=TARGET_VOCAB - TOKEN_OFFSET, min_frequency=2,
    initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()), special_tokens=[], show_progress=False)
def tokenizer_corpus():
    for row in train:
        yield row['question']
        yield row['code']
tokenizer.train_from_iterator(tokenizer_corpus(), trainer=trainer, length=2 * len(train))
VOCAB = tokenizer.get_vocab_size() + TOKEN_OFFSET
assert VOCAB == TARGET_VOCAB
TOKENIZER_FILE.write_text(tokenizer.to_str(pretty=False), encoding='utf-8')
tokenizer_hash = sha(TOKENIZER_FILE.read_bytes())

# The reversible ByteLevel alphabet, as in tokenizers/src/pre_tokenizers/byte_level.rs.
visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
byte_chars = {b: chr(b) for b in visible}
for index, b in enumerate(b for b in range(256) if b not in byte_chars):
    byte_chars[b] = chr(256 + index)
char_bytes = {char: b for b, char in byte_chars.items()}
assert set(char_bytes) == set(pre_tokenizers.ByteLevel.alphabet())
TOKEN_BYTES = [b'', b'', b''] + [bytes(char_bytes[c] for c in tokenizer.id_to_token(i))
                                for i in range(VOCAB - TOKEN_OFFSET)]
assert all(TOKEN_BYTES[i] for i in range(TOKEN_OFFSET, VOCAB))

def encode(text): return [i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids]
def decode_bytes(ids):
    parts = []
    for i in ids:
        if i == EOS: break
        if i >= TOKEN_OFFSET: parts.append(TOKEN_BYTES[i])
    return b''.join(parts)
def decode(ids): return decode_bytes(ids).decode('utf-8', errors='replace')
def length_stats(values):
    values = sorted(values)
    return {'count': len(values), 'mean': sum(values) / len(values), 'min': values[0],
        'p50': values[int(0.5 * (len(values) - 1))], 'p95': values[int(0.95 * (len(values) - 1))],
        'max': values[-1]}
def audit_tokenization(rows):
    report = {}
    for field in ('question', 'code'):
        byte_lengths, bpe_lengths = [], []
        for row in rows:
            text = row[field]
            ids = encode(text)
            assert decode_bytes(ids) == text.encode('utf-8'), 'Round-trip failed'
            assert tokenizer.decode([i - TOKEN_OFFSET for i in ids], skip_special_tokens=False) == text
            assert all(TOKEN_OFFSET <= i < VOCAB for i in ids)
            assert len(ids) + 2 <= MAX_POSITIONS
            byte_lengths.append(len(text.encode('utf-8')))
            bpe_lengths.append(len(ids))
        report[field] = {'utf8_bytes': length_stats(byte_lengths), 'bpe_tokens': length_stats(bpe_lengths),
            'total_bytes_per_bpe_token': sum(byte_lengths) / sum(bpe_lengths)}
    return report
probes = ['', '    return 1\n', '\n\n\t x\r\n', '  trailing  ',
          'Привет 世界 🧠', 'x = 0.005 + 123456789\n', '<pad><bos><eos>',
          ''.join(chr(i) for i in range(256))]
for text in probes: assert decode_bytes(encode(text)) == text.encode('utf-8')
reloaded_tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
assert all(reloaded_tokenizer.encode(t).ids == tokenizer.encode(t).ids for t in probes)
result['tokenization'] = {'kind': 'own byte-level BPE', 'target_model_vocabulary': TARGET_VOCAB,
    'actual_model_vocabulary': VOCAB, 'content_id_offset': TOKEN_OFFSET, 'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS},
    'normalizer': None, 'add_prefix_space': False, 'use_regex': True, 'min_frequency': 2,
    'training_corpus': 'only original20000 train questions and codes; no heldout/test',
    'training_corpus_sha256': train_hash, 'tokenizer_sha256': tokenizer_hash,
    'train_round_trip': audit_tokenization(train), 'validation64_round_trip': audit_tokenization(validation),
    'extra192_round_trip': audit_tokenization(validation_extra), 'probe_round_trip_count': len(probes),
    'training_and_audit_seconds': time.perf_counter() - tokenizer_started}
result['phase'] = 'tokenizer_ready'
save_results()
print('TOKENIZATION', json.dumps(result['tokenization']))
for text in ['def solve():\n    return 1\n', '    ', 'return', '12345']:
    print('TOKENS', repr(text), [(i, repr(TOKEN_BYTES[i])) for i in encode(text)])

def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS], dtype=torch.long) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels


## 4. Прежняя модель и KV-кеш

Новые веса, тот же набор Transformer-блоков. Проверка сравнит кешированные логиты с обычным decoder на разных длинах и PAD, а также короткую greedy-генерацию. После каждого контрольного обучения проверка повторится.


In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
TOKEN_BYTE_LENGTHS = torch.tensor([len(b) for b in TOKEN_BYTES], device=device)
config = {'seed': SEED, 'train_size': len(train), 'batch_size_train': BATCH_SIZE, 'total_steps': TOTAL_STEPS,
    'evaluation_steps': list(EVALUATION_STEPS), 'd_model': D_MODEL, 'encoder_layers': N_LAYERS,
    'decoder_layers': N_LAYERS, 'heads': 4, 'ffn_dim': 4 * D_MODEL, 'dropout': 0.1,
    'activation': 'ReLU', 'norm_first': False, 'lr': 0.001, 'optimizer': 'AdamW',
    'weight_decay': optimizer.param_groups[0]['weight_decay'], 'clip_norm': 1.0, 'scheduler': None,
    'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': VOCAB, 'content_id_offset': TOKEN_OFFSET,
    'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS}, 'tokenizer_sha256': tokenizer_hash,
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP,
    'batch_size_inference': INFERENCE_BATCH, 'inference': 'Post-LN incremental KV cache'}
result['config'] = config
result['parameters'] = sum(p.numel() for p in model.parameters())
assert result['parameters'] == 2241024
result['phase'] = 'model_ready'
save_results()
print('PARAMETERS', result['parameters'])


In [ ]:
class CachedDecoder:
    @torch.inference_mode()
    def __init__(self, model, src, capacity):
        model.eval()
        if not 0 < capacity <= model.positions.num_embeddings:
            raise ValueError('Invalid cache capacity')
        self.model, self.capacity, self.position = model, capacity, 0
        self.batch_size = len(src)
        self.width = model.tokens.embedding_dim
        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads
        self.head_dim = self.width // self.heads
        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).
        self.source_allowed = src.ne(PAD)[:, None, None, :]
        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)
        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))
        self.layers = []
        for layer in model.transformer.decoder.layers:
            if layer.norm_first:
                raise ValueError('This experiment implements the saved Post-LN model')
            for attention in (layer.self_attn, layer.multihead_attn):
                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads
                        or attention.bias_k is not None or attention.bias_v is not None
                        or attention.add_zero_attn):
                    raise ValueError('Unsupported attention configuration')
            attention = layer.multihead_attn
            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],
                                    attention.in_proj_bias[self.width:])
            cross_k, cross_v = projected_kv.chunk(2, dim=-1)
            shape = (len(src), self.heads, capacity, self.head_dim)
            self.layers.append({
                'self_k': memory.new_empty(shape), 'self_v': memory.new_empty(shape),
                'cross_k': self.split_heads(cross_k), 'cross_v': self.split_heads(cross_v),
            })

    def split_heads(self, values):
        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

    def attend(self, attention, q, k, v, allowed):
        # One query at the latest position; all supplied keys are from its past/current.
        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                                dropout_p=0.0, is_causal=False)
        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)
        return attention.out_proj(output)

    @torch.inference_mode()
    def step(self, token_ids):
        if self.position >= self.capacity:
            raise ValueError('Cache capacity exceeded')
        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:
            raise ValueError('One token per batch row is required')
        t = self.position
        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)
        x = x + self.position_vectors[t][None, None, :]
        self.valid[:, t] = token_ids.ne(PAD)
        self_allowed = self.valid[:, None, None, :t + 1]
        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):
            attention = layer.self_attn
            q, k, v = F.linear(x, attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)
            cache['self_k'][:, :, t:t + 1] = self.split_heads(k)
            cache['self_v'][:, :, t:t + 1] = self.split_heads(v)
            self_output = self.attend(attention, self.split_heads(q),
                cache['self_k'][:, :, :t + 1], cache['self_v'][:, :, :t + 1], self_allowed)
            x = layer.norm1(x + layer.dropout1(self_output))

            attention = layer.multihead_attn
            q = F.linear(x, attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])
            cross_output = self.attend(attention, self.split_heads(q),
                                      cache['cross_k'], cache['cross_v'], self.source_allowed)
            x = layer.norm2(x + layer.dropout2(cross_output))
            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
            x = layer.norm3(x + layer.dropout3(feed_forward))

        if self.model.transformer.decoder.norm is not None:
            x = self.model.transformer.decoder.norm(x)
        self.position += 1
        return self.model.head(x[:, 0])

@torch.inference_mode()
def generate_cached(model, src, max_new_tokens=512, max_output_bytes=None):
    model.eval()
    cache = CachedDecoder(model, src, max_new_tokens)
    tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
    done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
    byte_stopped = done.clone()
    output_bytes = torch.zeros(len(src), device=src.device, dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = cache.step(tgt[:, -1])
        logits[:, [PAD, BOS]] = -torch.inf
        token = logits.argmax(-1).masked_fill(done, PAD)
        if max_output_bytes is not None:
            overflow = (~done) & (token >= TOKEN_OFFSET) & (output_bytes + TOKEN_BYTE_LENGTHS[token] > max_output_bytes)
            # Stop BEFORE the whole overflowing BPE token; do not force a different token/EOS.
            byte_stopped |= overflow
            token = token.masked_fill(overflow, PAD)
            done |= overflow
        output_bytes += TOKEN_BYTE_LENGTHS[token]
        done |= token.eq(EOS)
        tgt = torch.cat([tgt, token[:, None]], dim=1)
        if bool(done.all()): break
    return tgt[:, 1:], byte_stopped

@torch.inference_mode()
def check_cached(rows, prefix_limit=64, greedy_cap=16):
    model.eval()
    src = source_batch([r['question'] for r in rows])
    tgt = pad_sequence([torch.tensor([BOS] + encode(r['code']) + [EOS]) for r in rows],
                       batch_first=True, padding_value=PAD).to(device)[:, :prefix_limit]
    memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
    cache = CachedDecoder(model, src, tgt.shape[1])
    positions = {0, 1, 2, 7, 15, 31, tgt.shape[1] - 1}
    checks = []
    for t in range(tgt.shape[1]):
        actual = cache.step(tgt[:, t])
        if t in positions:
            expected = model.head(model.decode_prefix(tgt[:, :t+1], memory, src.eq(PAD))[:, -1])
            checks.append({'position': t, 'max_abs_error': (actual - expected).abs().max().item(),
                'all_close': bool(torch.isclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL).all())})
    ordinary = model.generate(src, max_new_tokens=greedy_cap)
    cached, _ = generate_cached(model, src, max_new_tokens=greedy_cap)
    return {'positions': checks, 'atol': LOGIT_ATOL, 'rtol': LOGIT_RTOL,
        'all_close': all(c['all_close'] for c in checks),
        'greedy_equal': ordinary.shape == cached.shape and bool(torch.equal(ordinary, cached)),
        'greedy_cap': greedy_cap}
result['initial_cache_check'] = check_cached(validation[:3])
save_results()
print('CACHE_CHECK', json.dumps(result['initial_cache_check']))
assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal'], 'KV-кеш не совпал; сохрани ZIP'


## 5. Диагностика и checkpoints

Return считается внутри самой функции `solve`, без вложенных функций/классов. Генерации сохраняются целиком вместе с token IDs; некорректный UTF-8 отмечается отдельно. Reference-код не исполняется.


In [ ]:
@torch.inference_mode()
def evaluate_loss(rows):
    model.eval()
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
            ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

def code_properties(code):
    def scoped_nodes(node):
        stack = [node]
        while stack:
            current = stack.pop()
            yield current
            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
                stack.extend(ast.iter_child_nodes(current))
    try:
        tree = ast.parse(code)
        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'solve']
        has_return = any(isinstance(n, ast.Return) and n.value is not None
            for f in functions for stmt in f.body for n in scoped_nodes(stmt))
        return {'parses_as_python': True, 'defines_solve': bool(functions), 'solve_returns_value': has_return,
                'syntax_error_type': None, 'syntax_error_message': None, 'indentation_error': False}
    except (SyntaxError, ValueError, RecursionError) as error:
        return {'parses_as_python': False, 'defines_solve': False, 'solve_returns_value': False,
                'syntax_error_type': type(error).__name__, 'syntax_error_message': str(error),
                'indentation_error': isinstance(error, IndentationError)}
def inspect_ids(row, ids, byte_stopped=False):
    active = [i for i in ids if i != PAD]
    if EOS in active: active = active[:active.index(EOS) + 1]
    raw = decode_bytes(active)
    try:
        code = raw.decode('utf-8')
        valid_utf8 = True
    except UnicodeDecodeError:
        code, valid_utf8 = raw.decode('utf-8', errors='replace'), False
    eos = EOS in active
    example = {'question': row['question'], 'generated_code': code, 'generated_token_ids': active,
        'generated_tokens_including_eos': len(active), 'generated_utf8_bytes': len(raw), 'valid_utf8': valid_utf8,
        'eos_emitted': eos, 'reached_token_cap': len(active) == GENERATION_CAP,
        'token_cap_without_eos': len(active) == GENERATION_CAP and not eos,
        'byte_budget_stop': bool(byte_stopped),
        'stop_reason': 'eos' if eos else ('byte_budget' if byte_stopped else 'token_cap'),
        **code_properties(code)}
    if 'code' in row:
        example.update({'reference_code': row['code'], 'reference_utf8_bytes': len(row['code'].encode()),
            'reference_bpe_tokens_including_eos': len(encode(row['code'])) + 1,
            'exact_code_match': valid_utf8 and code == row['code'],
            'exact_tokens_with_eos': active == encode(row['code']) + [EOS]})
    return example

@torch.inference_mode()
def inspect_split(name, rows, max_output_bytes=None, with_loss=True):
    model.eval()
    fixed_loss = evaluate_loss(rows) if with_loss else None
    # Warm-up is outside generation timing and never changes weights.
    generate_cached(model, source_batch([rows[0]['question']]), max_new_tokens=8)
    sync()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        tokens, stopped = generate_cached(model, source_batch([r['question'] for r in batch]),
            max_new_tokens=GENERATION_CAP, max_output_bytes=max_output_bytes)
        examples.extend(inspect_ids(row, ids, stop) for row, ids, stop
                        in zip(batch, tokens.cpu().tolist(), stopped.cpu().tolist()))
    sync()
    seconds = time.perf_counter() - started
    summary = {'split': name, 'examples': len(rows), 'bpe_loss': fixed_loss, 'seconds': seconds,
        'answers_per_second': len(rows) / seconds,
        'generated_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples),
        'generated_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples),
        'mean_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples) / len(rows),
        'mean_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples) / len(rows),
        'max_new_tokens': GENERATION_CAP, 'max_output_bytes': max_output_bytes,
        'batch_size_inference': INFERENCE_BATCH, 'peak_gpu_memory_mib':
            torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
        'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading',
        'syntax_error_types': dict(Counter(e['syntax_error_type'] for e in examples if e['syntax_error_type']))}
    for metric in ['parses_as_python', 'defines_solve', 'solve_returns_value', 'indentation_error', 'valid_utf8',
                   'eos_emitted', 'reached_token_cap', 'token_cap_without_eos', 'byte_budget_stop']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
    if with_loss:
        for metric in ['exact_code_match', 'exact_tokens_with_eos']:
            summary[metric + '_count'] = sum(e[metric] for e in examples)
    summary['bpe_tokens_per_second'] = summary['generated_tokens_including_eos'] / seconds
    summary['output_bytes_per_second'] = summary['generated_utf8_bytes'] / seconds
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for e in examples[:3]: print('EXAMPLE', name, json.dumps(e, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'tokenizer_json': TOKENIZER_FILE.read_text(encoding='utf-8'), 'tokenizer_sha256': tokenizer_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}
    temporary = LATEST_CHECKPOINT.with_suffix('.pt.tmp')
    torch.save(state, temporary)
    temporary.replace(LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in EVALUATION_STEPS:
        name = f'step-{step:05d}.pt'
        target = RUN_DIR / name
        temporary = target.with_suffix('.pt.tmp')
        torch.save(state, temporary)
        temporary.replace(target)
        result['checkpoint_files'][str(step)] = name
    save_results()


## 6. Обучение с нуля до 50 000

На каждом контроле — прежние train64 и validation64. Дополнительные heldout192 и ограничение512 байтов — после финального шага. Таймер обучения исключает все эти оценки, печать и запись файлов. Повторный запуск ячейки на обученном optimizer запрещён.


In [ ]:
assert not optimizer.state and result['actual_steps'] == 0, 'Train уже запускался; сохрани файлы, не запускай эту ячейку повторно'
assert next(model.parameters()).dtype == torch.float32
result['phase'] = 'training'
result['training_timer_scope'] = 'batch encoding/construction, transfers, forward/backward, clipping, AdamW, block-end CUDA sync'
result['training_log'].append({'step': 0, 'train_batch_loss': None,
    'validation_loss_16': evaluate_loss(validation[:16]), 'training_seconds': 0.0})
save_results()
run_started = time.perf_counter()
training_seconds = 0.0
completed_steps = 0
sync()
block_started, block_active = time.perf_counter(), True
try:
    for step in range(1, TOTAL_STEPS + 1):
        model.train()
        src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        completed_steps = step
        if step == 1 or step % LOG_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            sync()
            training_seconds += time.perf_counter() - block_started
            block_active = False
            result['actual_steps'] = completed_steps
            result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
            train_loss, val_loss = loss.item(), evaluate_loss(validation[:16])
            if not math.isfinite(train_loss) or not math.isfinite(val_loss):
                raise FloatingPointError('Non-finite loss; last finite checkpoint is kept')
            entry = {'step': step, 'train_batch_loss': train_loss, 'validation_loss_16': val_loss,
                'training_seconds': training_seconds, 'updates_per_second': step / training_seconds,
                'mean_pair_presentations': step * BATCH_SIZE / len(train)}
            result['training_log'].append(entry)
            print('LOSS', json.dumps(entry), flush=True)
            save_results()
            if step in EVALUATION_STEPS:
                parity = check_cached(validation[:3])
                result['diagnostics'][str(step)] = {'cache_check': parity}
                save_results()
                if not parity['all_close'] or not parity['greedy_equal']:
                    raise RuntimeError('KV parity failed; download results instead of continuing')
                for name, rows in [('train64', train_diagnostic), ('validation64', validation)]:
                    result['diagnostics'][str(step)][name] = inspect_split(name, rows)
                    save_results()
            if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
                save_checkpoint(completed_steps)
                print('CHECKPOINT', completed_steps, flush=True)
            sync()
            if step < TOTAL_STEPS:
                block_started, block_active = time.perf_counter(), True
    result['run_finished'] = True
    result['phase'] = 'training_finished'
except BaseException as error:
    result['phase'] = 'training_stopped'
    result['training_error'] = {'type': type(error).__name__, 'message': str(error)}
    raise
finally:
    sync()
    if block_active: training_seconds += time.perf_counter() - block_started
    result['actual_steps'] = completed_steps
    result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
    result['active_wall_seconds_including_diagnostics_and_checkpoints'] = time.perf_counter() - run_started
    if result.get('training_error', {}).get('type') != 'FloatingPointError':
        save_checkpoint(completed_steps)
    save_results()


## 7. Дополнительная validation и прежний byte-limit

При byte-limit512 прекращаем генерацию **перед** BPE-токеном, который переполнил бы бюджет; не подменяем его EOS или другим токеном. Поэтому фактическая длина может быть немного меньше512 байтов. Это диагностика покрытия, а не идеально одинаковый поиск между разными токенизациями.


In [ ]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'final_validation'
result['final_validation'] = {}
for name, rows, byte_limit in [('validation192_extra', validation_extra, None),
                              ('validation64_byte512', validation, BYTE_BUDGET)]:
    result['final_validation'][name] = inspect_split(name, rows, max_output_bytes=byte_limit)
    save_results()
last = result['diagnostics'][str(TOTAL_STEPS)]['validation64']['summary']
print('VALIDATION64 versus saved byte50k:')
print('Byte50k AST/return/EOS:', 41, 40, 61, '| BPE:',
      last['parses_as_python_count'], last['solve_returns_value_count'], last['eos_emitted_count'])
print('BPE-loss is measured per BPE token; do not compare it numerically with byte-loss.')
result['phase'] = 'validation_finished'
save_results()


## 8. Полный test: все 1919 вопросов и таймер T4

Test скачивается после обучения, BPE на нём не обучается. Порядок строк исходный. Даже если получится дольше180 секунд, замер завершается полностью и сохраняется. Это не финальная посылка преподавателю.


In [ ]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'test_data'
save_results()
test_file = cache / 'test.parquet'
if not test_file.exists(): urlretrieve(f'{base_url}/data/test/test.parquet', test_file)
assert sha(test_file.read_bytes()) == '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a'
test_questions = pq.read_table(test_file, columns=['question'])['question'].to_pylist()
assert len(test_questions) == 1919
assert all(len(encode(q)) + 2 <= MAX_POSITIONS for q in test_questions)
test_rows = [{'question': q} for q in test_questions]
result['test_data'] = {'rows': len(test_rows), 'questions_sha256': data_hash(test_rows),
    'file_sha256': sha(test_file.read_bytes()), 'row_order': 'unchanged parquet order', 'dataset_revision': revision}
result['phase'] = 'test_generation'
save_results()
test_output = inspect_split('test1919', test_rows, with_loss=False)
TEST_GENERATIONS = RUN_DIR / 'test-generations.json'
write_json(TEST_GENERATIONS, test_output['examples'])
result['test'] = {'summary': test_output['summary'], 'examples_file': TEST_GENERATIONS.name}
result['test']['summary']['within_180_seconds'] = test_output['summary']['seconds'] <= 180
result['test_evaluated'] = True
result['phase'] = 'complete'
save_results()
print('TEST_SUMMARY', json.dumps(result['test']['summary']), flush=True)


## 9. Скачать архив — также после остановки предыдущей ячейки

ZIP содержит tokenizer, JSON с индивидуальными validation-генерациями, test-генерации, latest и все контрольные checkpoints. Сам notebook с outputs скачай отдельно через меню Colab.


In [ ]:
# This cell never trains, generates or restarts anything.
assert RESULTS_FILE.exists(), 'Сначала должны появиться результаты запуска'
result['archive_prepared'] = True
save_results()
README_FILE = RUN_DIR / 'README.txt'
README_FILE.write_text(
    'Experiment08: own BPE4k, original20k pairs, new2+2/128 FP32 model.\n'
    'results.json contains phase/completed steps/validation examples and test summary.\n'
    'tokenizer.json has4093 content IDs; model adds offset3 and PAD/BOS/EOS0/1/2.\n'
    'latest.pt and step-*.pt include model, optimizer, tokenizer JSON, data hashes and RNG.\n'
    'test-generations.json exists only after complete test. Programs were not executed.\n'
    'For analysis return this ZIP and the IPYNB downloaded with outputs.\n', encoding='utf-8')
files_to_pack = sorted(p for p in RUN_DIR.iterdir() if p.is_file()
                       and p.name != 'manifest.json' and not p.name.endswith('.tmp'))
manifest = {'experiment': result['experiment'], 'actual_steps': result['actual_steps'], 'phase': result['phase'],
    'files': [{'name': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in files_to_pack]}
write_json(RUN_DIR / 'manifest.json', manifest)
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in files_to_pack + [RUN_DIR / 'manifest.json']: archive.write(p, arcname=p.name)
print('ARCHIVE', str(ARCHIVE_FILE), ARCHIVE_FILE.stat().st_size, 'bytes', '| steps:', result['actual_steps'])
from google.colab import files
files.download(str(ARCHIVE_FILE))
